In [ ]:
import os
import re
import sys
import glob
import numpy as np
import matplotlib.pyplot as plt
modulePath = os.path.abspath(os.path.join('../..'))
if modulePath not in sys.path:
    sys.path.append(modulePath)
from common.unitConverter import UnitConverter as uc
plt.rcParams.update({
    "font.family" : "serif",
    "font.size" : 15,
    "mathtext.fontset" : "stix",
    "font.serif" : ['STIXGeneral']
})

In [ ]:
ReVec = [
    5, 10, 50, 100
]
folder = "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_force_evaluation/sphere_bgk_u003"

folderVec = [
    f"{folder}/sphere_dx20",
    f"{folder}/sphere_dx25",
    f"{folder}/sphere_dx32",
]

DLbVec = [
    20,
    25,
    32,
]

markerVec = [
    # "P",
    "X",
    'o',
    's',
    'd',
    '^',
    '*',
]

colorVec = [
    "#B700FF",
    "#05A361",
    "#D33737",
    "#001AFF",
    "#FFC629",
    "#119100",
    "#A17808",
]

ULb = 0.03

avgStartFactor = 0.8
stStartFactor = 0.3

labelVec = [
    "Present LBM BGK D/dx=20",
    "Present LBM BGK D/dx=25",
    "Present LBM BGK D/dx=32",
]

rhoPhy = 1.204
gamma = 1.0

dxVec = [1.0 / DLb for DLb in DLbVec]
ucVec = [uc(dx, rhoPhy, gamma) for dx in dxVec]

In [ ]:
def read_col_probe(filePath, colIdx, skipHeader=2):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, skip_header=skipHeader, usecols=colIdx)
        vec = vec[~np.isnan(vec)]
    f.close()
    return vec

def read_csv_col(filePath, colIdx, skipHeader=1):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, delimiter=',', skip_header=skipHeader, usecols=colIdx)
    f.close()
    return vec

def extract_reynolds_number(path):
    match = re.search(r"Re(\d+)", path)
    if match:
        return int(match.group(1))
    return 0

def cal_coef_sphere(force, area, rho, U):
    return force / (0.5 * rho * U**2 * area)

def cal_frontal_area_sphere(diameter):
    return np.pi * diameter**2 / 4


In [ ]:
ReVecVec = []
cdVecVec = []
clVecVec = []

stepVecVec = []
cxVecVec = []
cyVecVec = []
czVecVec = []

for case in range(0, len(folderVec)):
    pattern = f"{folderVec[case]}/sph_dx{int(DLbVec[case])}_Re*/forcetorque/forceSphere/forcetorque0000.txt"
    fileList = glob.glob(pattern)
    fileList = sorted(fileList, key=extract_reynolds_number)
    # print(fileList)
    ReVec = []
    avgCdVec = []
    avgClVec = []

    stepVec = []
    cxVec = []
    cyVec = []
    czVec = []
    for ff in range(0, len(fileList)):
        Re = extract_reynolds_number(fileList[ff])
        forceX = read_col_probe(fileList[ff], 2, 2)
        forceY = read_col_probe(fileList[ff], 3, 2)
        forceZ = read_col_probe(fileList[ff], 4, 2)
        idxStart = int(avgStartFactor * len(forceX))
        area = cal_frontal_area_sphere(DLbVec[case])
        cd = cal_coef_sphere(forceX, area, 1.0, ULb)
        cl = cal_coef_sphere(forceY, area, 1.0, ULb)
        cz = cal_coef_sphere(forceZ, area, 1.0, ULb)
        avgCd = np.mean(cd[idxStart:])
        avgCl = np.mean(cl[idxStart:])

        avgCdVec.append(avgCd)
        avgClVec.append(avgCl)
        ReVec.append(Re)

        stepCol = read_col_probe(fileList[ff], 0, 2)
        idxSt = int(stStartFactor * len(forceY))
        stepVec.append(stepCol[idxSt:])
        cxVec.append(cd[idxSt:])
        cyVec.append(cl[idxSt:])
        czVec.append(cz[idxSt:])
    
    ReVecVec.append(ReVec)
    cdVecVec.append(avgCdVec)
    clVecVec.append(avgClVec)

    stepVecVec.append(stepVec)
    cxVecVec.append(cxVec)
    cyVecVec.append(cyVec)
    czVecVec.append(czVec)



In [ ]:

#! Drag coefficient
refCdFileVec = [
    "/Users/maggie/repo/scripts/cases/force_evaluation/ref/cd_Re_sphere_liebster.csv",
    "/Users/maggie/repo/scripts/cases/force_evaluation/ref/cd_Re_sphere_johnson_patel.csv",
    "/Users/maggie/repo/scripts/cases/force_evaluation/ref/cd_Re_sphere_tau_lbm.csv",
]

refCdLabel = [
    'Liebster, Experiment',
    'Johnson & Patel, FVM/DNS',
    'Tao et al., LBM',
]

refCdLine = [
    '',
    ':',
    '--',
]

refCdMarker = [
    "o",
    "s",
    'D',
]

refReVec = []
refCdVec = []
for rr in range(0, len(refCdFileVec)):
    refReVec.append(read_csv_col(refCdFileVec[rr], 0, 1))
    refCdVec.append(read_csv_col(refCdFileVec[rr], 1, 1))



In [ ]:

figCd, axCd = plt.subplots(1, 1, figsize=(8, 6), facecolor='w', edgecolor='w')

for rr in range(0, len(refCdFileVec)):
    # axCd.loglog(refReVec[rr], refCdVec[rr], lw=1, linestyle=refCdLine[rr], c='grey', marker=refCdMarker[rr], ms=8, fillstyle='none', label=refCdLabel[rr])
    axCd.plot(refReVec[rr], refCdVec[rr], lw=1, linestyle=refCdLine[rr], c='grey', marker=refCdMarker[rr], ms=8, fillstyle='none', label=refCdLabel[rr])

for case in range(0, len(folderVec)):
    # axCd.loglog(ReVecVec[case], cdVecVec[case], lw=0, marker=markerVec[case], c=colorVec[case], ms=8, mew=1.5, fillstyle='none', label=labelVec[case])
    axCd.plot(ReVecVec[case], cdVecVec[case], lw=0, marker=markerVec[case], c=colorVec[case], ms=8, mew=1.5, fillstyle='none', label=labelVec[case])


axCd.legend(loc='upper right', frameon=False)
#! loglog
# axCd.set_xlim(0.01, 1000)
# axCd.set_ylim(0.01, 10000)

axCd.set_xlim(0, 120)
axCd.set_ylim(0, 10)
axCd.set_xlabel('Re')
axCd.set_ylabel(r'$C_d$')

In [ ]:
figC, axC = plt.subplots(3, 1, figsize=(10, 8), facecolor='w', edgecolor='w', sharex=True)

reynolds = 100
idxRe = int(np.abs(np.array(ReVec) - reynolds).argmin())
for case in range(0, len(folderVec)):
    axC[0].plot(stepVecVec[case][idxRe], cxVecVec[case][idxRe], label=labelVec[case], c=colorVec[case])
    axC[1].plot(stepVecVec[case][idxRe], cyVecVec[case][idxRe], label=labelVec[case], c=colorVec[case])
    axC[2].plot(stepVecVec[case][idxRe], czVecVec[case][idxRe], label=labelVec[case], c=colorVec[case])
    print(f"Case \"{labelVec[case]}\": Mean Cx={cdVecVec[case][idxRe]:.4f}, Mean Cy={clVecVec[case][idxRe]:.4f}")
    print(f"               CyMin={np.min(cyVecVec[case][idxRe])}, CyMax={np.max(cyVecVec[case][idxRe])}")

axC[0].set_ylabel(r'$C_x$')
axC[1].set_ylabel(r'$C_y$')
axC[2].set_ylabel(r'$C_z$')
axC[2].set_xlabel('Step')
axC[0].legend(loc='upper right', frameon=False)